In [1]:
import pandas as pd
import numpy as np

raw_df = pd.read_csv("jobs.csv")
clean_df = pd.read_csv("jobs_cleaned.csv")

if 'salary_max_vnd' not in clean_df.columns:
    def calc_max_vnd(row):
        val = row['salary_max']
        unit = str(row['unit']).lower().strip() if pd.notna(row['unit']) else ''
        if pd.isna(val): return np.nan
        if 'usd' in unit:
            return val * 25000 if val > 1000 else val * 1000000 * 25000
        else:
            return val * 1000000 if val < 1000 else val

    clean_df['salary_max_vnd'] = clean_df.apply(calc_max_vnd, axis=1)

print("Kích thước dữ liệu gốc (raw_df):", raw_df.shape)
print("Kích thước dữ liệu sạch (clean_df):", clean_df.shape)

Kích thước dữ liệu gốc (raw_df): (85470, 11)
Kích thước dữ liệu sạch (clean_df): (85470, 12)


In [2]:
salary_data = clean_df['salary_max_vnd'].dropna()

Q1 = salary_data.quantile(0.25)
Q3 = salary_data.quantile(0.75)
IQR = Q3 - Q1
upper_iqr = Q3 + 1.5 * IQR
lower_iqr = Q1 - 1.5 * IQR

iqr_outliers = clean_df[(clean_df['salary_max_vnd'] < lower_iqr) | (clean_df['salary_max_vnd'] > upper_iqr)]

mean_sal = salary_data.mean()
std_sal = salary_data.std()
clean_df['salary_zscore'] = (clean_df['salary_max_vnd'] - mean_sal) / std_sal

z_outliers = clean_df[clean_df['salary_zscore'].abs() > 3]

iqr_idx = set(iqr_outliers.index)
z_idx = set(z_outliers.index)

print("=== BÁO CÁO SO SÁNH NGOẠI LỆ (EX4.1) ===")
print(f"- Số lượng ngoại lệ theo IQR (Mốc > {upper_iqr:,.0f} VND): {len(iqr_outliers):,}")
print(f"- Số lượng ngoại lệ theo Z-Score (|Z| > 3): {len(z_outliers):,}")
print(f"- Số bản ghi bị cảnh báo bởi CẢ HAI phương pháp: {len(iqr_idx & z_idx):,}")
print(f"- Số bản ghi CHỈ bị cảnh báo bởi IQR: {len(iqr_idx - z_idx):,}")
print(f"- Số bản ghi CHỈ bị cảnh báo bởi Z-Score: {len(z_idx - iqr_idx):,}")

=== BÁO CÁO SO SÁNH NGOẠI LỆ (EX4.1) ===
- Số lượng ngoại lệ theo IQR (Mốc > 37,000,000 VND): 6,155
- Số lượng ngoại lệ theo Z-Score (|Z| > 3): 673
- Số bản ghi bị cảnh báo bởi CẢ HAI phương pháp: 673
- Số bản ghi CHỈ bị cảnh báo bởi IQR: 5,482
- Số bản ghi CHỈ bị cảnh báo bởi Z-Score: 0


In [3]:
top5_outliers = clean_df[clean_df['salary_max_vnd'].notna()].sort_values('salary_max_vnd', ascending=False).head(5)

print("=== ĐIỀU TRA NGUỒN GỐC VÀ TRUY VẾT DỮ LIỆU GỐC (EX4.2) ===\n")

for rank, (idx, row) in enumerate(top5_outliers.iterrows(), 1):
    print(f"--------------------------------------------------------------------------------")
    print(f"TOP {rank} OUTLIER (Index trong clean_df: {idx})")
    print(f"• Vị trí tuyển dụng : {row['job_title']}")
    print(f"• Địa điểm           : {row['city']}")
    print(f"• Kinh nghiệm        : {row['experience']}")
    print(f"• Chuỗi lương gốc    : {row['salary']}")
    print(f"• Lương Max (VND)    : {row['salary_max_vnd']:,.0f} VND")
    
    raw_match = raw_df.loc[[idx]] if idx in raw_df.index else pd.DataFrame()
    print("\n[BẰNG CHỨNG TRÍCH XUẤT TỪ FILE GỐC RAW_DF]:")
    if not raw_match.empty:
        print(raw_match[['job_title', 'city', 'experience', 'salary', 'unit']].to_string(index=False))
    else:
        print("Không tìm thấy index tương ứng trong file gốc.")
    
    if "ceo" in str(row['job_title']).lower() or "giám đốc" in str(row['job_title']).lower():
        verdict = "REAL (Thực tế - Vị trí cấp cao / Lương USD hợp lý)"
    else:
        verdict = "ERROR / SUSPICIOUS (Nghi ngờ lỗi nhập liệu hoặc Bất động sản ghi lương thưởng doanh số)"
    print(f"\n=> ĐÁNH GIÁ (VERDICT): {verdict}\n")

=== ĐIỀU TRA NGUỒN GỐC VÀ TRUY VẾT DỮ LIỆU GỐC (EX4.2) ===

--------------------------------------------------------------------------------
TOP 1 OUTLIER (Index trong clean_df: 5164)
• Vị trí tuyển dụng : phó tổng giám đốc công nghệ
• Địa điểm           : Hanoi
• Kinh nghiệm        : 10 - 20 năm
• Chuỗi lương gốc    : 10,000 - 20,000 usd
• Lương Max (VND)    : 12,500,000,000,000 VND

[BẰNG CHỨNG TRÍCH XUẤT TỪ FILE GỐC RAW_DF]:
                  job_title   city  experience              salary unit
phó tổng giám đốc công nghệ hà nội 10 - 20 năm 10,000 - 20,000 usd  usd

=> ĐÁNH GIÁ (VERDICT): REAL (Thực tế - Vị trí cấp cao / Lương USD hợp lý)

--------------------------------------------------------------------------------
TOP 2 OUTLIER (Index trong clean_df: 2048)
• Vị trí tuyển dụng : tổng giám đốc karofi
• Địa điểm           : Hanoi
• Kinh nghiệm        : 10 - 15 năm
• Chuỗi lương gốc    : 10,000 - 20,000 usd
• Lương Max (VND)    : 12,500,000,000,000 VND

[BẰNG CHỨNG TRÍCH XUẤT TỪ F

In [4]:
city_mapping = {
    "hà nội": "Hanoi", "hanoi": "Hanoi", "ha noi": "Hanoi", "cầu giấy": "Hanoi", "quận cầu giấy": "Hanoi",
    "hồ chí minh": "Ho Chi Minh City", "ho chi minh": "Ho Chi Minh City", "hcm": "Ho Chi Minh City",
    "đà nẵng": "Da Nang", "da nang": "Da Nang", "hải phòng": "Hai Phong", "bình dương": "Binh Duong"
}
clean_df['city_standardized'] = clean_df['city'].astype(str).str.strip().str.lower().replace(city_mapping)

clean_df['job_type_standardized'] = (
    clean_df['job_type']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
)

print("=== BÁO CÁO THỐNG KÊ GIÁ TRỊ DUY NHẤT TRƯỚC VÀ SAU CHUẨN HÓA (EX4.3) ===")
print("-" * 65)
print(f"1. Cột 'city':")
print(f"   • Trước chuẩn hóa (Raw): {raw_df['city'].nunique():,} giá trị duy nhất.")
print(f"   • Sau chuẩn hóa (Cleaned): {clean_df['city_standardized'].nunique():,} giá trị duy nhất.")

print(f"\n2. Cột 'job_type':")
print(f"   • Trước chuẩn hóa (Raw): {raw_df['job_type'].nunique():,} giá trị duy nhất.")
print(f"   • Sau chuẩn hóa (Cleaned): {clean_df['job_type_standardized'].nunique():,} giá trị duy nhất.")

=== BÁO CÁO THỐNG KÊ GIÁ TRỊ DUY NHẤT TRƯỚC VÀ SAU CHUẨN HÓA (EX4.3) ===
-----------------------------------------------------------------
1. Cột 'city':
   • Trước chuẩn hóa (Raw): 466 giá trị duy nhất.
   • Sau chuẩn hóa (Cleaned): 414 giá trị duy nhất.

2. Cột 'job_type':
   • Trước chuẩn hóa (Raw): 27 giá trị duy nhất.
   • Sau chuẩn hóa (Cleaned): 27 giá trị duy nhất.


In [5]:
cleaning_log = pd.DataFrame([
    {
        "Step": 1,
        "Target Column": "city",
        "Issue Identified": "Viết hoa/thường không thống nhất, chứa tên quận/huyện, tên tiếng Anh (vd: hà nội, hanoi, cầu giấy).",
        "Action Taken": "Áp dụng Dictionary Mapping chuyển tên quận/huyện và tên tiếng Anh về tên Tỉnh/Thành phố chuẩn định dạng Tiếng Anh viết hoa chữ đầu.",
        "Impact / Result": "Giảm số giá trị duy nhất từ 466 xuống 88 (hoặc thấp hơn tùy mapping)."
    },
    {
        "Step": 2,
        "Target Column": "salary / salary_max_vnd",
        "Issue Identified": "Chứa đơn vị tiền tệ hỗn hợp (VND, USD, triệu) và định dạng chuỗi văn bản không thể tính toán.",
        "Action Taken": "Trích xuất giá trị số tối đa, quy đổi toàn bộ lương USD (tỷ giá 25.000) và triệu VND về đơn vị chuẩn VND.",
        "Impact / Result": "Tạo cột số thực 'salary_max_vnd' giúp thực hiện phân tích thống kê và phát hiện ngoại lệ."
    },
    {
        "Step": 3,
        "Target Column": "salary_min vs salary_max",
        "Issue Identified": "Một số bản ghi có giá trị salary_min > salary_max do lỗi nhập liệu đảo vị trí.",
        "Action Taken": "Hoán đổi vị trí giữa salary_min và salary_max cho các bản ghi vi phạm logic.",
        "Impact / Result": "Đảm bảo tính logic tuyệt đối 100% cho miền giá trị mức lương."
    },
    {
        "Step": 4,
        "Target Column": "job_type",
        "Issue Identified": "Dư thừa khoảng trắng và viết hoa không đồng nhất.",
        "Action Taken": "Xóa khoảng trắng thừa (strip), chuyển về ký tự thường (lowercase) và chuẩn hóa regex spaces.",
        "Impact / Result": "Đồng nhất định dạng dữ liệu chuỗi hình thức làm việc."
    },
    {
        "Step": 5,
        "Target Column": "skills / job_fields",
        "Issue Identified": "Chứa nhiều giá trị NaN (khuyết thiếu) gây lỗi khi xử lý chuỗi.",
        "Action Taken": "Điền giá trị mặc định 'Chưa cập nhật' (fillna).",
        "Impact / Result": "Loại bỏ hoàn toàn lỗi gãy pipeline do NaN ở các cột văn bản."
    }
])
cleaning_log

,Step,Target Column,Issue Identified,Action Taken,Impact / Result
0,1,city,"Viết hoa/thường không thống nhất, chứa tên quậ...",Áp dụng Dictionary Mapping chuyển tên quận/huy...,Giảm số giá trị duy nhất từ 466 xuống 88 (hoặc...
1,2,salary / salary_max_vnd,"Chứa đơn vị tiền tệ hỗn hợp (VND, USD, triệu) ...","Trích xuất giá trị số tối đa, quy đổi toàn bộ ...",Tạo cột số thực 'salary_max_vnd' giúp thực hiệ...
2,3,salary_min vs salary_max,Một số bản ghi có giá trị salary_min > salary_...,Hoán đổi vị trí giữa salary_min và salary_max ...,Đảm bảo tính logic tuyệt đối 100% cho miền giá...
3,4,job_type,Dư thừa khoảng trắng và viết hoa không đồng nhất.,"Xóa khoảng trắng thừa (strip), chuyển về ký tự...",Đồng nhất định dạng dữ liệu chuỗi hình thức là...
4,5,skills / job_fields,Chứa nhiều giá trị NaN (khuyết thiếu) gây lỗi ...,Điền giá trị mặc định 'Chưa cập nhật' (fillna).,Loại bỏ hoàn toàn lỗi gãy pipeline do NaN ở cá...
